# B2-019-attention-transformers — Practice p08 — Solution

**Type:** constrained-coding · **Difficulty:** core · **Concepts:** attention-mask, causal-self-attention

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260808`  
**Qualified Book 1 prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:C6-pytorch`, `book1:C11-neural-training`  
**Remediation:** review the linked Book 1 units before continuing: [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb).

## Independent solution

The additive mask is zero for j<=i and negative infinity for j>i. Adding it before stable softmax makes every forbidden exponential exactly zero while normalizing only the allowed entries. The first row therefore has weight exactly one on key zero.

In [ ]:
import numpy as np
SEED = 20260808
ATOL = 1e-12
RTOL = 1e-12

def causal_attention_np(q, k, v):
    q = np.asarray(q, dtype=np.float64)
    k = np.asarray(k, dtype=np.float64)
    v = np.asarray(v, dtype=np.float64)
    if q.ndim != 2 or k.ndim != 2 or v.ndim != 2:
        raise ValueError("q, k, and v must be rank two")
    n = q.shape[0]
    if n == 0 or k.shape[0] != n or v.shape[0] != n or q.shape[1] != k.shape[1]:
        raise ValueError("equal nonempty sequence lengths and matching key widths are required")
    if not np.all(np.isfinite(q)) or not np.all(np.isfinite(k)) or not np.all(np.isfinite(v)):
        raise ValueError("inputs must be finite")
    allowed = np.tril(np.ones((n, n), dtype=bool))
    mask = np.where(allowed, 0.0, -np.inf)
    scores = q @ k.T / np.sqrt(q.shape[1])
    masked_scores = scores + mask
    shifted = masked_scores - np.max(masked_scores, axis=-1, keepdims=True)
    numerators = np.exp(shifted)
    weights = numerators / np.sum(numerators, axis=-1, keepdims=True)
    output = weights @ v
    return mask, weights, output

q = np.eye(3, dtype=np.float64)
k = np.eye(3, dtype=np.float64)
v = np.arange(6, dtype=np.float64).reshape(3, 2)
mask, weights, output = causal_attention_np(q, k, v)
forbidden = np.triu(np.ones((3, 3), dtype=bool), k=1)
EXPECTED_WEIGHTS = np.array([[1.0, 0.0, 0.0], [0.3595425243193725, 0.6404574756806275, 0.0], [0.26445846149561975, 0.26445846149561975, 0.47108307700876034]], dtype=np.float64)
EXPECTED_OUTPUT = np.array([[0.0, 1.0], [1.280914951361255, 2.2809149513612548], [2.4132492310262808, 3.4132492310262808]], dtype=np.float64)

### Answer check

In [ ]:
assert mask.shape == weights.shape == (3, 3)
assert output.shape == (3, 2)
assert mask.dtype == weights.dtype == output.dtype == np.float64
assert np.count_nonzero(weights[forbidden]) == 0
np.testing.assert_allclose(weights, EXPECTED_WEIGHTS, atol=ATOL, rtol=RTOL)
np.testing.assert_allclose(output, EXPECTED_OUTPUT, atol=ATOL, rtol=RTOL)
np.testing.assert_allclose(weights.sum(axis=-1), np.ones(3), atol=ATOL, rtol=RTOL)